In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

import pandas as pd
import numpy as np

from tqdm import tqdm
from PIL import Image
from collections import defaultdict

import torch
import glob
import os
import re

In [2]:
# 1. Load model with multimodal support
model_id = "./typhoon-ocr-2b"
llm = LLM(
    model                  = model_id,
    gpu_memory_utilization = 0.8,
    max_model_len          = 32000,
    limit_mm_per_prompt    = {"image": 4},     # ← very important: how many images per prompt
    enforce_eager          = True,
    quantization           = "bitsandbytes",
)

INFO 03-22 02:32:38 [utils.py:238] non-default args: {'max_model_len': 32000, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 4}, 'model': './typhoon-ocr-2b'}
INFO 03-22 02:32:38 [model.py:531] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 03-22 02:32:38 [model.py:1554] Using max model len 32000
INFO 03-22 02:32:39 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-22 02:32:40 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-22 02:32:40 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-22 02:32:40 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-22 02:32:40 [vllm.py:957] Cudagraph is disabled under eager mode
(Engin

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1957346) INFO 03-22 02:32:50 [gpu_model_runner.py:4338] Model loading took 1.88 GiB memory and 4.376513 seconds
(EngineCore_DP0 pid=1957346) INFO 03-22 02:32:50 [gpu_model_runner.py:5254] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore_DP0 pid=1957346) INFO 03-22 02:32:55 [gpu_worker.py:424] Available KV cache memory: 27.65 GiB
(EngineCore_DP0 pid=1957346) INFO 03-22 02:32:55 [kv_cache_utils.py:1314] GPU KV cache size: 258,848 tokens
(EngineCore_DP0 pid=1957346) INFO 03-22 02:32:55 [kv_cache_utils.py:1319] Maximum concurrency for 32,000 tokens per request: 8.09x
(EngineCore_DP0 pid=1957346) INFO 03-22 02:32:55 [core.py:282] init engine (profile, create kv cache, warmup model) took 4.84 seconds
(EngineCore_DP0 pid=1957346) INFO 03-22 02:32:55 [vllm.py:957] Cudagraph is disabled under eager mode
WARNING 03-22 02:32:55 [vllm.py:781] Enforce eager set, disabling torch.compile and C

## Prompt prep

In [3]:
data = np.sort(glob.glob("./final_data/images/*.png"))
grouped_data = defaultdict(list)

for path in data:
    # Updated regex to match either 'constituency' OR 'party_list'
    # followed by the digits pattern
    matches = re.search(r'((?:constituency|party_list)_\d+_\d+)', path)
    
    if matches:
        group_id = matches.group(1)
        grouped_data[group_id].append(path)

# Convert to a standard dict for viewing
result = dict(grouped_data)
print(f"Total groups found: {len(result.keys())}")

Total groups found: 300


In [4]:
# Convert your numpy array/list to a string for the prompt
party_list_str = ", ".join(['ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต', 'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย', 'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่', 'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ', 'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง', 'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ', 'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม', 'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน', 'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ', 'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย', 'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม', 'สร้างชาติ', 'ใหม่', 'พลังสังคมใหม่', 'สร้างอนาคตไทย', 'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่', 'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่', 'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธรรมใหม่', 'กรีน', 'แผ่นดินธรรม', 'ประชาไทย', 'ประชาอาสาชาติ', 'เครือข่ายชาวนาแห่งประเทศไทย', 'ไทยรวมไทย', 'พลังไทยรักชาติ'])

chat_template = '''## Task: Multi-Page OCR and Data Extraction for Thai Election Results (Form ส.ส. ๖/๑).

## Context:
You are provided with images from document {{fname}}. 

## Extraction Rules:
1. **Identify Table:** Extract rows from the table containing candidate names and parties.
2. **Party Name Validation:** You MUST match the extracted party name against the **Allowed Party List** below. If the OCR is slightly off, correct it to the closest match from this list:
   [{party_list_str}]
3. **Vote Extraction (Rightmost Column):** - The "ได้คะแนน" column contains both Thai digits and Thai words in parentheses, e.g., "๓๔,๑๖๗ (สามหมื่น...)".
   - You must extract ONLY the numerical value.
   - Convert Thai digits (๑๒๓) to Arabic integers (123).
   - Ignore the Thai text inside the parentheses.
4. **ID & Order:**
   - doc_id: "{fname}"
5. **Exclusion:** Skip the "รวมคะแนนทั้งสิ้น" (Total) row.

## Output Format: Strictly JSON array.
[{{
    "doc_id": "{fname}",
    "party_name": "string",
    "votes": number
}}]
'''

In [5]:
processor = AutoProcessor.from_pretrained(model_id, use_fast=True)

documents = list(result.keys())
conversations = []
for doc in tqdm(documents):
    # 1. Prepare the messages in the format the processor expects
    messages = [
        {
            "role": "user",
            "content": [
                # For every image in your list, add an entry
                *[{"type": "image"} for _ in result[doc]],
                {"type": "text", "text": chat_template.format(party_list_str=party_list_str, fname=doc)},
            ]
        }
    ]

    # 2. Apply the template via the PROCESSOR
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 3. Store for vLLM or standard HF inference
    conversations.append({
        "prompt": prompt_text,
        "multi_modal_data": {"image": [Image.open(r) for r in result[doc]]}
    })

print(conversations[0])

100%|██████████| 300/300 [00:10<00:00, 27.87it/s]

{'prompt': '<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|>## Task: Multi-Page OCR and Data Extraction for Thai Election Results (Form ส.ส. ๖/๑).\n\n## Context:\nYou are provided with images from document {fname}. \n\n## Extraction Rules:\n1. **Identify Table:** Extract rows from the table containing candidate names and parties.\n2. **Party Name Validation:** You MUST match the extracted party name against the **Allowed Party List** below. If the OCR is slightly off, correct it to the closest match from this list:\n   [ประชาธิปัตย์, ภูมิใจไทย, เศรษฐกิจ, กล้าธรรม, พลวัต, ประชาชน, เพื่อไทย, ไทยภักดี, รวมไทยสร้างชาติ, ปวงชนไทย, ไทยสร้างไทย, โอกาสใหม่, วิชชั่นใหม่, ประชาธิปไตยใหม่, รักชาติ, ไทยก้าวใหม่, ทางเลือกใหม่, พลังประชารัฐ, ประชากรไทย, ความหวังใหม่, อนาคตไทย, บ้านเมือง, เพื่อบ้านเมือง, ไทยก้าวหน้า, รวมไทยสร้าง, แรงงานสร้างชาติ, ไทยชนะ, พร้อม, ไทยพิทักษ์ธรรม, ไทยก้าวไทย, ไทยก้าวใหม, เส

## Inference

In [6]:
# 3. Generate (batched version accepts list of conversations)
sampling_params = SamplingParams(
    temperature=0.0, top_p=1, top_k=-1,
    max_tokens=16000,
    skip_special_tokens=True,
)

outputs = llm.generate(conversations[0], sampling_params=sampling_params)

for o in outputs:
    generated_text = o.outputs[0].text
    print(generated_text)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

{"doc_id": "constituency_10_1",
"party_name": "พรรคการเมือง",
"votes": "๓๔,๑๖๗ (สามหมื่นสี่พันหนึ่งร้อยหกสิบเจ็ด)\n๑๔,๘๑๓ (หนึ่งหมื่นสี่พันแปดร้อยสิบสาม)\n๑๔,๓๖๘ (หนึ่งหมื่นสี่พันสามร้อยหกสิบแปด)\n๖,๐๓๐ (หกพันสามสิบ)\n๒,๐๗๕ (สองพันเจ็ดสิบห้า)\n๑,๑๓๓ (หนึ่งพันหนึ่งร้อยสามสิบสาม)\n๑,๐๒๓ (หนึ่งพันยี่สิบสาม)\n๙๗๙ (เก้าร้อยเจ็ดสิบเก้า)\n๖๒๙ (หกร้อยยี่สิบเก้า)\n๔๘๙ (สี่ร้อยแปดสิบเก้า)\n๓๕๑ (สามร้อยห้าสิบเอ็ด)\n๒๔๔ (สองร้อยสี่สิบสี่)\n๑๖๘ (หนึ่งร้อยหกสิบแปด)\n๑๕๔ (หนึ่งร้อยห้าสิบสี่)\n๑๑๓ (หนึ่งร้อยสิบสาม)\n๙๔ (เก้าสิบสี่)\n๘๐ (แปดสิบ)\n๗๗,๐๗๕ (เจ็ดหมื่นเจ็ดพันเจ็ดสิบห้า)\n๘๒,๔๒๑ บัตร (แปดหมื่นสองพันสี่ร้อยยี่สิบเอ็ด)\n๔. ผลการนับคะแนน\n๔.๑ บัตรดี\n๔.๒ บัตรเสีย\n๔.๓ บัตรไม่เลือกผู้สมัครผู้ใด\n๕. จำนวนคะแนนที่ผู้สมัครรับเลือกตั้งแต่ละคนได้รับเรียงตามลำดับผู้ได้คะแนนสูงสุด\n๑๑ จำนวนผู้มีสิทธิเลือกตั้ง\n๑.๑ จำนวนผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อผู้มีสิทธิเลือกตั้ง ๑๓๐,๔๔๕ คน (หนึ่งแสนสามหมื่นสี่ร้อยสี่สิบห้า)\n๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน ๘๒,๔๒๑ คน (แปดหมื่นสองพันสี่ร้อยยี่สิบเอ็ด)\n

In [7]:
submission = pd.read_csv("final_data/submission_template_v3.csv", encoding='utf-8-sig')